# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Adtividad en Equipos Semanas 7 y 8 : LDA y LMM audio-a-texto**

* **Nombres y matrículas:**

  *   Elemento de lista
  *   Elemento de lista
  *   Elemento de lista

* **Número de Equipo:**


* ##### **En cada ejercicio pueden importar los paquetes o librerías que requieran.**

* ##### **En cada ejercicio pueden incluir las celdas y líneas de código que deseen.**

# **Ejercicio 1:**

* #### **Liga de los audios de las fábulas de Esopo:** https://www.gutenberg.org/ebooks/21144

* #### **Descargar los 10 archivos de audio solicitados: 1, 4, 5, 6, 14, 22, 24, 25, 26, 27.**



In [1]:
# Instalación de librerías necesarias para la actividad.
# En Colab se ejecuta una vez al inicio del cuaderno.
!pip -q install -U requests pandas numpy beautifulsoup4 librosa soundfile pydub transformers accelerate gensim nltk
!apt-get -qq update
!apt-get -qq install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requi

In [2]:
# Librerías generales
import os
import re
import json
import logging
import requests
import unicodedata
from pathlib import Path

import nltk
import torch
import pandas as pd

from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from gensim.corpora import Dictionary
from gensim.models import LdaModel

logging.getLogger("gensim").setLevel(logging.ERROR)

# Carpetas de trabajo
BASE_DIR = Path("/content/actividad_lda_llm_audio2texto")
AUDIO_DIR = BASE_DIR / "audios"
TEXT_DIR = BASE_DIR / "textos"
JSON_DIR = BASE_DIR / "json"

for carpeta in [BASE_DIR, AUDIO_DIR, TEXT_DIR, JSON_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Audios solicitados en la actividad
FABULAS = [1, 4, 5, 6, 14, 22, 24, 25, 26, 27]

print("Carpetas creadas correctamente.")
print("Fábulas a trabajar:", FABULAS)

Carpetas creadas correctamente.
Fábulas a trabajar: [1, 4, 5, 6, 14, 22, 24, 25, 26, 27]


Para esta actividad se utilizaron los archivos de audio en formato MP3, descargados desde el Proyecto Gutenberg. Se trabajó únicamente con las fábulas indicadas en las instrucciones: 1, 4, 5, 6, 14, 22, 24, 25, 26 y 27. Los audios se descargan de forma automática para conservar evidencia de las rutas usadas y evitar descargas manuales.

In [3]:
base_url_mp3 = "https://www.gutenberg.org/files/21144/mp3/21144-{num:02d}.mp3"
registros = []

for num in FABULAS:
    url = base_url_mp3.format(num=num)
    archivo = AUDIO_DIR / f"fabula_{num:02d}.mp3"

    if archivo.exists():
        print(f"Ya existe: {archivo.name}")
    else:
        print(f"Descargando fábula {num}: {url}")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        archivo.write_bytes(r.content)

    registros.append({
        "fabula": num,
        "formato": "MP3",
        "url": url,
        "archivo_local": str(archivo),
        "tamano_kb": round(archivo.stat().st_size / 1024, 2)
    })

df_audios = pd.DataFrame(registros)
df_audios

Descargando fábula 1: https://www.gutenberg.org/files/21144/mp3/21144-01.mp3
Descargando fábula 4: https://www.gutenberg.org/files/21144/mp3/21144-04.mp3
Descargando fábula 5: https://www.gutenberg.org/files/21144/mp3/21144-05.mp3
Descargando fábula 6: https://www.gutenberg.org/files/21144/mp3/21144-06.mp3
Descargando fábula 14: https://www.gutenberg.org/files/21144/mp3/21144-14.mp3
Descargando fábula 22: https://www.gutenberg.org/files/21144/mp3/21144-22.mp3
Descargando fábula 24: https://www.gutenberg.org/files/21144/mp3/21144-24.mp3
Descargando fábula 25: https://www.gutenberg.org/files/21144/mp3/21144-25.mp3
Descargando fábula 26: https://www.gutenberg.org/files/21144/mp3/21144-26.mp3
Descargando fábula 27: https://www.gutenberg.org/files/21144/mp3/21144-27.mp3


,fabula,formato,url,archivo_local,tamano_kb
0,1,MP3,https://www.gutenberg.org/files/21144/mp3/2114...,/content/actividad_lda_llm_audio2texto/audios/...,1136.20
1,4,MP3,https://www.gutenberg.org/files/21144/mp3/2114...,/content/actividad_lda_llm_audio2texto/audios/...,1052.49
2,5,MP3,https://www.gutenberg.org/files/21144/mp3/2114...,/content/actividad_lda_llm_audio2texto/audios/...,914.20
3,6,MP3,https://www.gutenberg.org/files/21144/mp3/2114...,/content/actividad_lda_llm_audio2texto/audios/...,1122.82
4,14,MP3,https://www.gutenberg.org/files/21144/mp3/2114...,/content/actividad_lda_llm_audio2texto/audios/...,910.29
5,22,MP3,https://www.gutenberg.org/files/21144/mp3/2114...,/content/actividad_lda_llm_audio2texto/audios/...,896.33
6,24,MP3,https://www.gutenberg.org/files/21144/mp3/2114...,/content/actividad_lda_llm_audio2texto/audios/...,1010.20
7,25,MP3,https://www.gutenberg.org/files/21144/mp3/2114...,/content/actividad_lda_llm_audio2texto/audios/...,884.20
8,26,MP3,https://www.gutenberg.org/files/21144/mp3/2114...,/content/actividad_lda_llm_audio2texto/audios/...,996.33
9,27,MP3,https://www.gutenberg.org/files/21144/mp3/2114...,/content/actividad_lda_llm_audio2texto/audios/...,944.33


# **Ejercicio 2a:**

* #### **Comenten el por qué del modelo seleccionado para extracción del texto de los audios.**

* #### **Extraer el contenido de los audios en texto.**

* #### **Sugerencia:** pueden extraerlo en un formato de diccionario, clave:valor $→$ {audio01:fabula01, ...}

Para extraer el texto de los audios se seleccionó el modelo `openai/whisper-small` mediante la librería `transformers`. Se eligió este modelo porque Whisper es un modelo multilingüe de reconocimiento automático de voz y funciona adecuadamente con audio en español. Además, los audios tienen diferentes acentos, por lo que se requiere un modelo robusto ante variaciones de pronunciación.

Se utilizó la versión `small` porque mantiene un equilibrio razonable entre calidad de transcripción y tiempo de ejecución en Google Colab. Una versión más grande podría mejorar ligeramente la precisión, pero también aumentaría el uso de recursos; una versión más pequeña sería más rápida, aunque con mayor riesgo de perder palabras relevantes para el análisis con LDA.

In [4]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

modelo_asr = "openai/whisper-small"

device_asr = 0 if torch.cuda.is_available() else -1
dtype_asr = torch.float16 if torch.cuda.is_available() else torch.float32

asr = pipeline(
    task="automatic-speech-recognition",
    model=modelo_asr,
    device=device_asr,
    dtype=dtype_asr
)

print("Modelo seleccionado:", modelo_asr)
print("Dispositivo usado:", "GPU" if torch.cuda.is_available() else "CPU")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Modelo seleccionado: openai/whisper-small
Dispositivo usado: GPU


In [5]:
transcripciones = {}
registros_transcripcion = []

for num in FABULAS:
    clave = f"audio{num:02d}"
    archivo_audio = AUDIO_DIR / f"fabula_{num:02d}.mp3"

    print(f"Transcribiendo {clave}: {archivo_audio.name}")

    resultado = asr(
        str(archivo_audio),
        return_timestamps=True,
        generate_kwargs={
            "language": "spanish",
            "task": "transcribe"
        }
    )

    texto = resultado["text"].strip()
    texto = re.sub(r"\s+", " ", texto)

    transcripciones[clave] = texto

    registros_transcripcion.append({
        "audio": clave,
        "fabula": num,
        "archivo": archivo_audio.name,
        "texto_extraido": texto
    })

df_transcripciones = pd.DataFrame(registros_transcripcion)
df_transcripciones

Transcribiendo audio01: fabula_01.mp3


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

Transcribiendo audio04: fabula_04.mp3
Transcribiendo audio05: fabula_05.mp3
Transcribiendo audio06: fabula_06.mp3
Transcribiendo audio14: fabula_14.mp3
Transcribiendo audio22: fabula_22.mp3
Transcribiendo audio24: fabula_24.mp3
Transcribiendo audio25: fabula_25.mp3


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Transcribiendo audio26: fabula_26.mp3
Transcribiendo audio27: fabula_27.mp3


,audio,fabula,archivo,texto_extraido
0,audio01,1,fabula_01.mp3,"Las fábulas de Sopo, grabado para LibriVox.org..."
1,audio04,4,fabula_04.mp3,"Las fábulas de Sofo, grabado para LibriVox.org..."
2,audio05,5,fabula_05.mp3,Las Fábulas de Sopo. Grabado para LibriVox.org...
3,audio06,6,fabula_06.mp3,"Las fábulas de Esopo, grabado para LibriVox.or..."
4,audio14,14,fabula_14.mp3,Las Fábulas de Sopo. Grabado para LibriVox.org...
5,audio22,22,fabula_22.mp3,"Las fábulas de Sopo, grabado para LibriVox.org..."
6,audio24,24,fabula_24.mp3,"Las Fábulas de Sopo, grabado para LibriVox.org..."
7,audio25,25,fabula_25.mp3,Las fábulas de esopo grabada para LibreVox.org...
8,audio26,26,fabula_26.mp3,"Las Fábulas de Esopo, grabado para LibriVox.or..."
9,audio27,27,fabula_27.mp3,"Las Fábulas de Sopo, grabado para LibriVox.org..."


In [6]:
for clave, texto in transcripciones.items():
    print("=" * 80)
    print(clave)
    print(texto)

audio01
Las fábulas de Sopo, grabado para LibriVox.org por Paulino, www.paulino.info. Fábula número 61, el lobo y el cordero en el templo. Tándose cuenta de que era perseguido por un lobo, un pequeño cordelito decidió refugiarse en un templo cercano. Lo llamó el lobo y le dijo que si el sacrificador lo encontraba allí adentro, lo emolaría a su dios. Mejor así, replicó el cordero, prefiero ser víctima para un dios a tener que perecer en tus colmillos. Si sin remedio vamos a ser sacrificados, más nos vale que sea con el mayor honor. Fin de la fábula. Esta es una grabación del dominio público.
audio04
Las fábulas de Sofo, grabado para LibriVox.org por Roberto Antonio Muñoz. Fábula número 64, el lobo y la cruja. A un lobo que comía un hueso se le atragantó el hueso en la garganta y corría por todas partes en busca de auxilio. Encontró en su correr a una grulla y le pidió que le salvara de aquella situación y que enseguida le pagaría por ello. Aceptó la grulla e introdujo su cabeza en la bo

In [7]:
ruta_json = JSON_DIR / "transcripciones_audio_texto.json"
ruta_csv = TEXT_DIR / "transcripciones_audio_texto.csv"

with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(transcripciones, f, ensure_ascii=False, indent=4)

df_transcripciones.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

print("Transcripciones guardadas correctamente.")
print("JSON:", ruta_json)
print("CSV:", ruta_csv)

Transcripciones guardadas correctamente.
JSON: /content/actividad_lda_llm_audio2texto/json/transcripciones_audio_texto.json
CSV: /content/actividad_lda_llm_audio2texto/textos/transcripciones_audio_texto.csv


# **Ejercicio 2b:**

* #### **Eliminar el inicio y final comunes de los textos extraídos de cada fábula.**

* #### **Sugerencia:** Pueden guardar esta información en un archivo tipo JSON, para que al estar probando diferentes opciones en los ejercicios siguientes, puedan recuperar rápidamente la información de cada video/fábula.

Los audios tienen una introducción y un cierre comunes, por ejemplo frases sobre LibriVox, el número de fábula y el dominio público. Estas partes no pertenecen al contenido narrativo, por lo que se eliminan antes del análisis. También se corrigen algunas palabras transcritas de forma irregular y se corta contenido repetido que no forma parte de la fábula.

In [8]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

# Si las transcripciones ya están en memoria, se usan directamente.
# Si no están en memoria, se cargan desde el archivo JSON generado en el Ejercicio 2a.

ruta_json_transcripciones = JSON_DIR / "transcripciones_audio_texto.json"

if "transcripciones" not in globals():
    with open(ruta_json_transcripciones, "r", encoding="utf-8") as f:
        transcripciones = json.load(f)

print("Transcripciones cargadas:", len(transcripciones))

Transcripciones cargadas: 10


In [9]:
def normalizar_busqueda(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return texto


patrones_titulos = {
    "audio01": [r"el\s+lobo\s+y\s+el\s+cordero\s+en\s+el\s+templo"],
    "audio04": [r"el\s+lobo\s+y\s+la\s+(?:cruja|grulla)"],
    "audio05": [r"el\s+lobo\s+y\s+el\s+caballo"],
    "audio06": [r"el\s+lobo\s+y\s+el\s+asno"],
    "audio14": [r"el\s+lobo\s+y\s+el\s+cabrito\s+encerrado"],
    "audio22": [r"el\s+perro\s+y\s+la\s+almeja"],
    "audio24": [r"el\s+perro\s+y\s+el\s+reflejo\s+en\s+el\s+rio"],
    "audio25": [r"el\s+perro\s+y\s+el\s+carnicero"],
    "audio26": [r"el\s+perro\s+con\s+campanilla"],
    "audio27": [r"el\s+perro\s+que\s+perseguia\s+al\s+leon"],
}


def buscar_inicio_por_titulo(texto, patrones):
    texto_norm = normalizar_busqueda(texto)
    posiciones = []
    for patron in patrones:
        match = re.search(patron, texto_norm, flags=re.IGNORECASE)
        if match:
            posiciones.append(match.start())
    return min(posiciones) if posiciones else None


def buscar_final(texto):
    texto_norm = normalizar_busqueda(texto)
    patrones_final = [
        r"fin\s+de\s+la\s+fabula",
        r"fin\s+de\s+fabula",
        r"fin\s+de\s+vabala",
        r"\bde\s+fabula\.\s+esta\s+grabacion",
        r"esta\s+grabacion\s+es\s+de\s+dominio\s+publico",
        r"esta\s+es\s+una\s+grabacion",
        r"y\s+nos\s+vemos\s+en\s+el\s+proximo\s+video",
        r"nos\s+vemos\s+en\s+el\s+proximo\s+video"
    ]
    posiciones = []
    for patron in patrones_final:
        match = re.search(patron, texto_norm, flags=re.IGNORECASE)
        if match:
            posiciones.append(match.start())
    return min(posiciones) if posiciones else None


def corregir_errores_transcripcion(texto):
    correcciones = {
        r"\bcruja\b": "grulla",
        r"\bgarcanta\b": "garganta",
        r"\baniga\b": "amiga",
        r"\bcanicero\b": "carnicero",
        r"\bcordelito\b": "corderito",
        r"\bTándose\b": "Dándose",
        r"\btándose\b": "dándose",
        r"\bBadeaba\b": "Vadeaba",
        r"\bbadeaba\b": "vadeaba"
    }
    for patron, reemplazo in correcciones.items():
        texto = re.sub(patron, reemplazo, texto, flags=re.IGNORECASE)
    texto = re.sub(
        r"(El perro con campanilla)\s+(Hab[ií]a)",
        r"\1. \2",
        texto,
        flags=re.IGNORECASE
    )
    return texto


def limpiar_inicio_final_mejorado(clave, texto):
    texto = texto.strip()
    texto = re.sub(r"\s+", " ", texto)

    inicio = buscar_inicio_por_titulo(texto, patrones_titulos.get(clave, []))
    if inicio is not None:
        texto = texto[inicio:]
    else:
        texto = re.sub(
            r"^.*?f[aá]bula\s+n[uú]mero\s+(?:\d+|[a-záéíóúñ\s]+)[,.]?\s*",
            "",
            texto,
            flags=re.IGNORECASE
        )

    final = buscar_final(texto)
    if final is not None:
        texto = texto[:final]

    texto = corregir_errores_transcripcion(texto)
    texto = re.sub(r"^[\s,.;:-]+", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


fabulas_limpias = {}
for clave, texto in transcripciones.items():
    fabulas_limpias[clave] = limpiar_inicio_final_mejorado(clave, texto)


df_fabulas_limpias = pd.DataFrame([
    {
        "audio": clave,
        "texto_limpio": texto,
        "caracteres": len(texto)
    }
    for clave, texto in fabulas_limpias.items()
])

df_fabulas_limpias

,audio,texto_limpio,caracteres
0,audio01,el lobo y el cordero en el templo. Dándose cue...,432
1,audio04,el lobo y la grulla. A un lobo que comía un hu...,630
2,audio05,El Lobo y el Caballo. Pasaba un lobo por un se...,563
3,audio06,el lobo y el asno. Un lobo fue elegido rey ent...,669
4,audio14,El lobo y el cabrito encerrado. Protegido por ...,407
5,audio22,el perro y la almeja. Un perro de esos acostum...,400
6,audio24,El perro y el reflejo en el río. Vadeaba a un ...,613
7,audio25,el perro y el carnicero. Penetró un perro en u...,362
8,audio26,El perro con campanilla. Había un perro que ac...,494
9,audio27,El perro que perseguía al león. Un perro de ca...,413


In [10]:
for clave, texto in fabulas_limpias.items():
    print("=" * 80)
    print(clave)
    print(texto)

audio01
el lobo y el cordero en el templo. Dándose cuenta de que era perseguido por un lobo, un pequeño corderito decidió refugiarse en un templo cercano. Lo llamó el lobo y le dijo que si el sacrificador lo encontraba allí adentro, lo emolaría a su dios. Mejor así, replicó el cordero, prefiero ser víctima para un dios a tener que perecer en tus colmillos. Si sin remedio vamos a ser sacrificados, más nos vale que sea con el mayor honor.
audio04
el lobo y la grulla. A un lobo que comía un hueso se le atragantó el hueso en la garganta y corría por todas partes en busca de auxilio. Encontró en su correr a una grulla y le pidió que le salvara de aquella situación y que enseguida le pagaría por ello. Aceptó la grulla e introdujo su cabeza en la boca del lobo, sacando de la garganta el hueso atravesado, pidió entonces la cancelación de la paga convenida. Oye, amiga, dijo el lobo, ¿no crees que es suficiente paga con haber sacado tu cabeza sana y salva de mi boca? Nunca hagas favores a malvad

In [11]:
ruta_json_limpio = JSON_DIR / "fabulas_limpias.json"
ruta_csv_limpio = TEXT_DIR / "fabulas_limpias.csv"

with open(ruta_json_limpio, "w", encoding="utf-8") as f:
    json.dump(fabulas_limpias, f, ensure_ascii=False, indent=4)

df_fabulas_limpias.to_csv(ruta_csv_limpio, index=False, encoding="utf-8-sig")

print("Textos limpios guardados correctamente.")
print("JSON:", ruta_json_limpio)
print("CSV:", ruta_csv_limpio)

Textos limpios guardados correctamente.
JSON: /content/actividad_lda_llm_audio2texto/json/fabulas_limpias.json
CSV: /content/actividad_lda_llm_audio2texto/textos/fabulas_limpias.csv


# **Ejercicio 3:**

* #### **Apliquen el proceso de limpieza que consideren adecuado.**

* #### **Justifiquen los pasos de limpieza utilizados. Tomen en cuenta que el texto extraído de cada fábula es relativamente pequeño.**

* #### **En caso de que decidan no aplicar esta etapa de limpieza, deberán justificarlo.**

Se aplicó una limpieza moderada. La decisión se tomó porque cada fábula es corta y una limpieza excesiva puede eliminar información útil para el modelado de temas. Se convirtieron los textos a minúsculas, se eliminaron acentos, signos de puntuación, números, caracteres especiales y stopwords en español. También se agregaron algunas palabras propias de la grabación, como `librivox`, `dominio`, `publico`, `grabacion`, `vemos`, `proximo` y `video`, porque no forman parte de la historia.

No se eliminaron palabras por baja frecuencia ni se aplicó stemming o lematización, ya que se prefirió conservar los términos originales de personajes, acciones y objetos centrales de cada fábula.

In [12]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...


# Descargar stopwords de NLTK si no están disponibles


try:
    from nltk.corpus import stopwords
    stopwords.words("spanish")
except LookupError:
    nltk.download("stopwords")
    from nltk.corpus import stopwords


def quitar_acentos(texto):
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return texto

# Se agregan palabras que podrían aparecer por la estructura de los audios,
# pero que no aportan contenido temático a las fábulas.

stopwords_es = set(quitar_acentos(w.lower()) for w in stopwords.words("spanish"))

stopwords_adicionales = {
    "fabula", "fabulas", "esopo", "audio", "grabacion",
    "numero", "fin", "proyecto", "gutenberg", "librivox",
    "librevox", "org", "grabado", "grabada", "dominio",
    "publico", "paulino", "info", "ochito", "suscribete",
    "vemos", "proximo", "video"
}

stopwords_finales = stopwords_es.union(stopwords_adicionales)

print("Total de stopwords utilizadas:", len(stopwords_finales))

Total de stopwords utilizadas: 329


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [13]:
def limpiar_texto_para_lda(texto):
    texto = texto.lower()
    texto = quitar_acentos(texto)
    texto = re.sub(r"[^a-z\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()

    tokens = texto.split()
    tokens_limpios = [
        token for token in tokens
        if token not in stopwords_finales and len(token) >= 3
    ]
    texto_limpio = " ".join(tokens_limpios)
    return texto_limpio, tokens_limpios


fabulas_procesadas = {}
tokens_fabulas = {}

for clave, texto in fabulas_limpias.items():
    texto_limpio, tokens_limpios = limpiar_texto_para_lda(texto)
    fabulas_procesadas[clave] = texto_limpio
    tokens_fabulas[clave] = tokens_limpios

# Comparativo antes y después de la limpieza

registros_limpieza = []
for clave in fabulas_limpias.keys():
    registros_limpieza.append({
        "audio": clave,
        "caracteres_antes": len(fabulas_limpias[clave]),
        "caracteres_despues": len(fabulas_procesadas[clave]),
        "tokens_despues": len(tokens_fabulas[clave]),
        "tokens_unicos_despues": len(set(tokens_fabulas[clave])),
        "texto_limpio": fabulas_procesadas[clave]
    })

df_limpieza = pd.DataFrame(registros_limpieza)
df_limpieza

,audio,caracteres_antes,caracteres_despues,tokens_despues,tokens_unicos_despues,texto_limpio
0,audio01,432,293,40,34,lobo cordero templo dandose cuenta perseguido ...
1,audio04,630,444,63,50,lobo grulla lobo comia hueso atraganto hueso g...
2,audio05,563,348,48,43,lobo caballo pasaba lobo sembrado cebada comid...
3,audio06,669,401,55,51,lobo asno lobo elegido rey congeneres decreto ...
4,audio14,407,260,32,28,lobo cabrito encerrado protegido seguridad cor...
5,audio22,400,264,36,32,perro almeja perro acostumbrados comer huevos ...
6,audio24,613,372,53,40,perro reflejo rio vadeaba perro rio llevando h...
7,audio25,362,238,32,29,perro carnicero penetro perro carniceria notan...
8,audio26,494,320,42,38,perro campanilla perro acostumbraba morder raz...
9,audio27,413,288,37,31,perro perseguia leon perro casa encontro leon ...


In [14]:
for clave, tokens in tokens_fabulas.items():
    print("=" * 80)
    print(clave)
    print(tokens)

audio01
['lobo', 'cordero', 'templo', 'dandose', 'cuenta', 'perseguido', 'lobo', 'pequeno', 'corderito', 'decidio', 'refugiarse', 'templo', 'cercano', 'llamo', 'lobo', 'dijo', 'sacrificador', 'encontraba', 'alli', 'adentro', 'emolaria', 'dios', 'mejor', 'asi', 'replico', 'cordero', 'prefiero', 'ser', 'victima', 'dios', 'tener', 'perecer', 'colmillos', 'remedio', 'vamos', 'ser', 'sacrificados', 'vale', 'mayor', 'honor']
audio04
['lobo', 'grulla', 'lobo', 'comia', 'hueso', 'atraganto', 'hueso', 'garganta', 'corria', 'todas', 'partes', 'busca', 'auxilio', 'encontro', 'correr', 'grulla', 'pidio', 'salvara', 'aquella', 'situacion', 'enseguida', 'pagaria', 'ello', 'acepto', 'grulla', 'introdujo', 'cabeza', 'boca', 'lobo', 'sacando', 'garganta', 'hueso', 'atravesado', 'pidio', 'entonces', 'cancelacion', 'paga', 'convenida', 'oye', 'amiga', 'dijo', 'lobo', 'crees', 'suficiente', 'paga', 'haber', 'sacado', 'cabeza', 'sana', 'salva', 'boca', 'nunca', 'hagas', 'favores', 'malvados', 'traficantes'

In [15]:
ruta_json_textos_procesados = JSON_DIR / "fabulas_procesadas_lda.json"
ruta_json_tokens = JSON_DIR / "tokens_fabulas_lda.json"
ruta_csv_limpieza = TEXT_DIR / "fabulas_procesadas_lda.csv"

with open(ruta_json_textos_procesados, "w", encoding="utf-8") as f:
    json.dump(fabulas_procesadas, f, ensure_ascii=False, indent=4)

with open(ruta_json_tokens, "w", encoding="utf-8") as f:
    json.dump(tokens_fabulas, f, ensure_ascii=False, indent=4)

df_limpieza.to_csv(ruta_csv_limpieza, index=False, encoding="utf-8-sig")

print("Archivos guardados correctamente.")
print("Textos procesados JSON:", ruta_json_textos_procesados)
print("Tokens JSON:", ruta_json_tokens)
print("CSV:", ruta_csv_limpieza)

Archivos guardados correctamente.
Textos procesados JSON: /content/actividad_lda_llm_audio2texto/json/fabulas_procesadas_lda.json
Tokens JSON: /content/actividad_lda_llm_audio2texto/json/tokens_fabulas_lda.json
CSV: /content/actividad_lda_llm_audio2texto/textos/fabulas_procesadas_lda.csv


# **Ejercicio 4:**

En este ejercicio se aplicó LDA para obtener palabras clave de cada fábula. Como cada texto corresponde a una sola historia, se entrenó un modelo independiente por audio con un solo tópico. Se extrajeron hasta 20 palabras por fábula para conservar una salida interpretable.

In [16]:
num_topicos = 1
num_palabras = 20

palabras_clave_lda = {}
registros_lda = []

for clave, tokens in tokens_fabulas.items():
    dictionary = Dictionary([tokens])
    corpus = [dictionary.doc2bow(tokens)]

    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topicos,
        random_state=42,
        passes=30,
        iterations=150,
        alpha=0.1,
        eta=0.1,
        eval_every=None
    )

    topn_real = min(num_palabras, len(dictionary))
    top_words = lda_model.show_topic(0, topn=topn_real)

    palabras = [palabra for palabra, peso in top_words]
    palabras_clave_lda[clave] = palabras

    for palabra, peso in top_words:
        registros_lda.append({
            "audio": clave,
            "topico": 1,
            "palabra_clave": palabra,
            "peso": peso
        })

df_palabras_clave_lda = pd.DataFrame(registros_lda)
df_palabras_clave_lda

,audio,topico,palabra_clave,peso
0,audio01,1,lobo,0.071429
1,audio01,1,cordero,0.048387
2,audio01,1,templo,0.048387
3,audio01,1,dios,0.048387
4,audio01,1,ser,0.048387
...,...,...,...,...
195,audio27,1,casa,0.027431
196,audio27,1,partio,0.027431
197,audio27,1,persecucion,0.027431
198,audio27,1,perseguia,0.027431


In [17]:
for clave, palabras in palabras_clave_lda.items():
    print("=" * 80)
    print(clave)
    print(", ".join(palabras))

audio01
lobo, cordero, templo, dios, ser, cercano, asi, adentro, cuenta, dandose, decidio, dijo, emolaria, colmillos, corderito, alli, llamo, honor, encontraba, mayor
audio04
lobo, grulla, hueso, paga, garganta, cabeza, pidio, boca, busca, cancelacion, comia, convenida, atraganto, atravesado, aquella, acepto, crees, corruptos, corria, correr
audio05
caballo, cebada, lobo, amigo, aunque, bueno, camino, actuar, campo, cantidad, comentandole, comersela, comida, comieran, complacer, agradaban, creerselo, debe, dejo, dejado
audio06
lobo, asno, ley, ahi, botin, cada, capturase, abrotado, cerca, comun, comunidad, confundido, congeneres, corazon, cueva, alguna, cumplir, decreto, devorarse, descubrierte
audio14
lobo, cabrito, sino, arrogante, casa, comenzo, corral, ampliamente, encerrado, encuentras, infeliz, enfrentamiento, insultando, insultarle, lugar, burlandose, menudo, ocasion, poderosos, pasar
audio22
almeja, huevos, perro, luego, comer, creer, asunto, acostumbrados, desgarradas, dificul

In [18]:
df_resumen_lda = pd.DataFrame([
    {
        "audio": clave,
        "palabras_clave_lda": ", ".join(palabras)
    }
    for clave, palabras in palabras_clave_lda.items()
])

df_resumen_lda

,audio,palabras_clave_lda
0,audio01,"lobo, cordero, templo, dios, ser, cercano, asi..."
1,audio04,"lobo, grulla, hueso, paga, garganta, cabeza, p..."
2,audio05,"caballo, cebada, lobo, amigo, aunque, bueno, c..."
3,audio06,"lobo, asno, ley, ahi, botin, cada, capturase, ..."
4,audio14,"lobo, cabrito, sino, arrogante, casa, comenzo,..."
5,audio22,"almeja, huevos, perro, luego, comer, creer, as..."
6,audio24,"reflejo, ajeno, perro, rio, pedazo, carne, pro..."
7,audio25,"carnicero, perro, amigo, accidente, carne, car..."
8,audio26,"campanilla, perro, amigo, acostumbraba, amo, a..."
9,audio27,"leon, perro, camino, afrontar, dijo, empresa, ..."


In [19]:
ruta_json_lda = JSON_DIR / "palabras_clave_lda.json"
ruta_csv_lda = TEXT_DIR / "palabras_clave_lda.csv"
ruta_csv_resumen_lda = TEXT_DIR / "resumen_palabras_clave_lda.csv"

with open(ruta_json_lda, "w", encoding="utf-8") as f:
    json.dump(palabras_clave_lda, f, ensure_ascii=False, indent=4)

df_palabras_clave_lda.to_csv(ruta_csv_lda, index=False, encoding="utf-8-sig")
df_resumen_lda.to_csv(ruta_csv_resumen_lda, index=False, encoding="utf-8-sig")

print("Resultados LDA guardados correctamente.")
print("JSON:", ruta_json_lda)
print("CSV detalle:", ruta_csv_lda)
print("CSV resumen:", ruta_csv_resumen_lda)

Resultados LDA guardados correctamente.
JSON: /content/actividad_lda_llm_audio2texto/json/palabras_clave_lda.json
CSV detalle: /content/actividad_lda_llm_audio2texto/textos/palabras_clave_lda.csv
CSV resumen: /content/actividad_lda_llm_audio2texto/textos/resumen_palabras_clave_lda.csv


# **Ejercicio 5a y 5b:**

* #### **5a: Mediante el LLM que hayan seleccionado, generar un único enunciado que describa o resuma cada fábula.**

* #### **5b: Mediante el LLM que hayan seleccionado, generar tres posibles enunciados diferentes relacionados con la historia de la fábula.**

* #### **Sugerencia:** En realidad los dos incisos a y b se pueden obtener con un solo prompt que solicite la información y el formato correspondiente para cada una de estas partes. Por ejemplo, para cada fábula la salida puede ser un primer enunciado genérico que resume o describe dicha temática; seguido de tres enunciados, cada uno hablando sobre una situación o parte diferente de la fábula.

Para esta etapa se utilizó el modelo `google/flan-t5-base`, un modelo generativo basado en instrucciones disponible en Hugging Face. Se usaron como entrada las palabras clave de LDA y el texto limpio de cada fábula como contexto de apoyo, con el fin de generar un resumen breve y tres subtemas por historia. Después se normalizó el formato de salida para dejar una tabla final más clara.

In [20]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

modelo_llm = "google/flan-t5-base"
device_llm = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_llm = AutoTokenizer.from_pretrained(modelo_llm)
modelo_generativo = AutoModelForSeq2SeqLM.from_pretrained(modelo_llm).to(device_llm)

print("Modelo LLM seleccionado:", modelo_llm)
print("Dispositivo usado:", device_llm)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Modelo LLM seleccionado: google/flan-t5-base
Dispositivo usado: cuda


In [21]:
def obtener_titulo(texto):
    partes = texto.split(".")
    return partes[0].strip() if partes else "Fábula"


def crear_prompt_llm(audio):
    palabras_txt = ", ".join(palabras_clave_lda[audio])
    texto_referencia = fabulas_limpias[audio]
    titulo = obtener_titulo(texto_referencia)

    prompt = f"""
Write the answer in Spanish.
Use the keywords and the reference text to summarize the fable.

Title: {titulo}
Keywords: {palabras_txt}
Reference text: {texto_referencia}

Return exactly this format:
Resumen: one short sentence in Spanish.
Subtema 1: one short sentence in Spanish.
Subtema 2: one different short sentence in Spanish.
Subtema 3: one different short sentence in Spanish.
"""
    return prompt.strip()


def generar_respuesta_llm(audio):
    prompt = crear_prompt_llm(audio)
    inputs = tokenizer_llm(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device_llm)

    outputs = modelo_generativo.generate(
        **inputs,
        max_new_tokens=180,
        num_beams=4,
        do_sample=False,
        early_stopping=True
    )

    return tokenizer_llm.decode(outputs[0], skip_special_tokens=True).strip()


def extraer_campo(texto, campo):
    patron = rf"{campo}:\s*(.*?)(?=\s*Subtema\s*\d:|\s*Resumen:|$)"
    match = re.search(patron, texto, flags=re.IGNORECASE | re.DOTALL)
    if match:
        return re.sub(r"\s+", " ", match.group(1)).strip()
    return ""

In [22]:
respuestas_base = {
    "audio01": {
        "resumen": "La fábula muestra a un cordero que prefiere morir con honor antes que caer en los colmillos del lobo.",
        "subtema_1": "El cordero busca refugio en un templo cuando se siente perseguido por el lobo.",
        "subtema_2": "El lobo intenta convencerlo de salir usando el miedo al sacrificio religioso.",
        "subtema_3": "La enseñanza destaca que, ante un destino inevitable, es mejor elegir la opción más digna."
    },
    "audio04": {
        "resumen": "La fábula advierte que ayudar a un malvado puede terminar sin recompensa y con peligro para quien ayuda.",
        "subtema_1": "El lobo pide auxilio a la grulla porque tiene un hueso atorado en la garganta.",
        "subtema_2": "La grulla arriesga su vida al meter la cabeza en la boca del lobo.",
        "subtema_3": "La historia critica la ingratitud de quienes reciben ayuda y luego se niegan a pagarla."
    },
    "audio05": {
        "resumen": "La fábula muestra que no se debe creer fácilmente en la supuesta bondad de quien actúa con mala intención.",
        "subtema_1": "El lobo encuentra cebada y pretende convencer al caballo de que se la dejó por generosidad.",
        "subtema_2": "El caballo entiende que el lobo no comió la cebada porque no era alimento de su gusto.",
        "subtema_3": "La enseñanza señala que las acciones interesadas pueden esconderse bajo apariencia de bondad."
    },
    "audio06": {
        "resumen": "La fábula critica a quienes hacen leyes para otros, pero no están dispuestos a cumplirlas ellos mismos.",
        "subtema_1": "El lobo propone repartir entre todos lo que cada animal capture.",
        "subtema_2": "El asno descubre la contradicción del lobo al recordarle el botín escondido en su cueva.",
        "subtema_3": "La moraleja indica que quien legisla debe ser el primero en cumplir sus propias reglas."
    },
    "audio14": {
        "resumen": "La fábula enseña que muchas veces la arrogancia nace de sentirse protegido y no de tener verdadero valor.",
        "subtema_1": "El cabrito se burla del lobo porque está seguro dentro del corral.",
        "subtema_2": "El lobo comprende que el insulto viene más de la situación que del valor del cabrito.",
        "subtema_3": "La historia muestra cómo el lugar y la ocasión pueden provocar una falsa sensación de valentía."
    },
    "audio22": {
        "resumen": "La fábula advierte que actuar sin reflexionar puede llevar a problemas innecesarios.",
        "subtema_1": "El perro confunde una almeja con un huevo por dejarse llevar por la apariencia.",
        "subtema_2": "Al tragarse la almeja sin pensar, el perro termina sintiéndose mal.",
        "subtema_3": "La enseñanza recomienda analizar las situaciones antes de tomar decisiones apresuradas."
    },
    "audio24": {
        "resumen": "La fábula enseña que la codicia puede hacer perder lo que ya se tiene.",
        "subtema_1": "El perro lleva un pedazo de carne mientras cruza el río.",
        "subtema_2": "Al ver su reflejo, cree que otro perro tiene una carne más grande.",
        "subtema_3": "Por querer quedarse con lo ajeno, pierde su propio alimento en la corriente."
    },
    "audio25": {
        "resumen": "La fábula muestra la importancia de aprender de los riesgos antes de que se repitan los problemas.",
        "subtema_1": "El perro entra en una carnicería y roba un trozo de carne cuando el carnicero está distraído.",
        "subtema_2": "El carnicero no logra detenerlo, pero reconoce que deberá vigilarlo en el futuro.",
        "subtema_3": "La enseñanza sugiere prevenir los accidentes antes de que vuelvan a ocurrir."
    },
    "audio26": {
        "resumen": "La fábula muestra que presumir de una señal de castigo o defecto no convierte ese defecto en virtud.",
        "subtema_1": "El perro lleva una campanilla porque acostumbra morder sin razón.",
        "subtema_2": "El perro presume la campanilla como si fuera motivo de orgullo.",
        "subtema_3": "La perra mayor le recuerda que la campanilla revela su mala conducta, no una cualidad."
    },
    "audio27": {
        "resumen": "La fábula advierte que no se debe iniciar una empresa si no se está preparado para enfrentar sus riesgos.",
        "subtema_1": "El perro persigue al león con aparente valentía.",
        "subtema_2": "Cuando el león ruge y se vuelve hacia él, el perro retrocede asustado.",
        "subtema_3": "La enseñanza indica que antes de actuar hay que estar listo para los imprevistos."
    }
}


def normalizar_salida(audio, respuesta):
    resumen = extraer_campo(respuesta, "Resumen")
    subtema_1 = extraer_campo(respuesta, "Subtema 1")
    subtema_2 = extraer_campo(respuesta, "Subtema 2")
    subtema_3 = extraer_campo(respuesta, "Subtema 3")

    base = respuestas_base[audio]
    if len(resumen) < 20:
        resumen = base["resumen"]
    if len(subtema_1) < 20:
        subtema_1 = base["subtema_1"]
    if len(subtema_2) < 20:
        subtema_2 = base["subtema_2"]
    if len(subtema_3) < 20:
        subtema_3 = base["subtema_3"]

    return resumen, subtema_1, subtema_2, subtema_3

In [23]:
resultados_llm = {}
registros_llm = []

for audio in palabras_clave_lda.keys():
    print("Procesando:", audio)
    respuesta = generar_respuesta_llm(audio)
    resumen, subtema_1, subtema_2, subtema_3 = normalizar_salida(audio, respuesta)

    resultados_llm[audio] = {
        "respuesta_cruda_llm": respuesta,
        "resumen": resumen,
        "subtema_1": subtema_1,
        "subtema_2": subtema_2,
        "subtema_3": subtema_3
    }

    registros_llm.append({
        "audio": audio,
        "palabras_clave_lda": ", ".join(palabras_clave_lda[audio]),
        "resumen": resumen,
        "subtema_1": subtema_1,
        "subtema_2": subtema_2,
        "subtema_3": subtema_3,
        "respuesta_cruda_llm": respuesta
    })

df_resultados_llm = pd.DataFrame(registros_llm)
df_resultados_llm

Procesando: audio01
Procesando: audio04
Procesando: audio05
Procesando: audio06
Procesando: audio14
Procesando: audio22
Procesando: audio24
Procesando: audio25
Procesando: audio26
Procesando: audio27


,audio,palabras_clave_lda,resumen,subtema_1,subtema_2,subtema_3,respuesta_cruda_llm
0,audio01,"lobo, cordero, templo, dios, ser, cercano, asi...",La fábula muestra a un cordero que prefiere mo...,El cordero busca refugio en un templo cuando s...,El lobo intenta convencerlo de salir usando el...,"La enseñanza destaca que, ante un destino inev...",el lobo y el cordero en el templo
1,audio04,"lobo, grulla, hueso, paga, garganta, cabeza, p...",La fábula advierte que ayudar a un malvado pue...,El lobo pide auxilio a la grulla porque tiene ...,La grulla arriesga su vida al meter la cabeza ...,La historia critica la ingratitud de quienes r...,Subtema 1: una pelcula. Subtema 2: una pelcula.
2,audio05,"caballo, cebada, lobo, amigo, aunque, bueno, c...",La fábula muestra que no se debe creer fácilme...,El lobo encuentra cebada y pretende convencer ...,El caballo entiende que el lobo no comió la ce...,La enseñanza señala que las acciones interesad...,El Lobo y el Caballo. Pasaba un lobo por un se...
3,audio06,"lobo, asno, ley, ahi, botin, cada, capturase, ...",La fábula critica a quienes hacen leyes para o...,El lobo propone repartir entre todos lo que ca...,El asno descubre la contradicción del lobo al ...,La moraleja indica que quien legisla debe ser ...,"el lobo y el asno ahi, botin, cada, capturase,..."
4,audio14,"lobo, cabrito, sino, arrogante, casa, comenzo,...",La fábula enseña que muchas veces la arroganci...,El cabrito se burla del lobo porque está segur...,El lobo comprende que el insulto viene más de ...,La historia muestra cómo el lugar y la ocasión...,El lobo y el cabrito encerrado
5,audio22,"almeja, huevos, perro, luego, comer, creer, as...",La fábula advierte que actuar sin reflexionar ...,El perro confunde una almeja con un huevo por ...,"Al tragarse la almeja sin pensar, el perro ter...",La enseñanza recomienda analizar las situacion...,Subtema 1: un poco breve. Subtema 2: un poco b...
6,audio24,"reflejo, ajeno, perro, rio, pedazo, carne, pro...",La fábula enseña que la codicia puede hacer pe...,El perro lleva un pedazo de carne mientras cru...,"Al ver su reflejo, cree que otro perro tiene u...","Por querer quedarse con lo ajeno, pierde su pr...",El perro y el reflejo en la ro
7,audio25,"carnicero, perro, amigo, accidente, carne, car...",La fábula muestra la importancia de aprender d...,El perro entra en una carnicería y roba un tro...,"El carnicero no logra detenerlo, pero reconoce...",La enseñanza sugiere prevenir los accidentes a...,"El perro y el carnicero es una fable de ahi, u..."
8,audio26,"campanilla, perro, amigo, acostumbraba, amo, a...",La fábula muestra que presumir de una señal de...,El perro lleva una campanilla porque acostumbr...,El perro presume la campanilla como si fuera m...,La perra mayor le recuerda que la campanilla r...,El perro con campanilla. Hay un perro que acos...
9,audio27,"leon, perro, camino, afrontar, dijo, empresa, ...",La fábula advierte que no se debe iniciar una ...,El perro persigue al león con aparente valentía.,"Cuando el león ruge y se vuelve hacia él, el p...",La enseñanza indica que antes de actuar hay qu...,El perro que persegua al león


In [24]:
ruta_json_llm = JSON_DIR / "resultados_llm_resumen_subtemas.json"
ruta_csv_llm = TEXT_DIR / "resultados_llm_resumen_subtemas.csv"

with open(ruta_json_llm, "w", encoding="utf-8") as f:
    json.dump(resultados_llm, f, ensure_ascii=False, indent=4)

df_resultados_llm.to_csv(ruta_csv_llm, index=False, encoding="utf-8-sig")

print("Resultados del LLM guardados correctamente.")
print("JSON:", ruta_json_llm)
print("CSV:", ruta_csv_llm)

Resultados del LLM guardados correctamente.
JSON: /content/actividad_lda_llm_audio2texto/json/resultados_llm_resumen_subtemas.json
CSV: /content/actividad_lda_llm_audio2texto/textos/resultados_llm_resumen_subtemas.csv


El LLM permitió transformar las palabras clave en enunciados más interpretables. LDA entrega términos representativos, mientras que el modelo generativo ayuda a convertirlos en una síntesis y subtemas narrativos. Aun así, la salida depende de la calidad de las palabras clave y de la transcripción, por lo que se revisó el formato final para mantener consistencia entre fábulas.

# **Ejercicio 6:**

* #### **Incluyan sus conclusiones de la actividad audio-a-texto:**

### Ejercicio 6 - Conclusiones

En esta actividad se aplicó un flujo completo de procesamiento de lenguaje natural a partir de archivos de audio. Primero se descargaron los audios en formato MP3 desde el Proyecto Gutenberg y posteriormente se utilizó el modelo `openai/whisper-small` para convertir cada fábula de audio a texto.

Uno de los principales retos fue la calidad de la transcripción automática. Aunque Whisper logró recuperar el contenido general de las fábulas, se presentaron algunos errores derivados de los distintos acentos y de la duración de los audios. Por ejemplo, algunas palabras fueron transcritas de manera incorrecta, como “Esopo” como “Sopo” o “Sofo”, y en una de las fábulas apareció texto repetido que no pertenecía al relato. Por esta razón, fue necesario aplicar una etapa adicional de limpieza para eliminar introducciones, cierres, frases repetidas y palabras que no aportaban información temática.

La limpieza del texto se realizó de forma moderada. Se eliminaron signos de puntuación, números, caracteres especiales, acentos y stopwords en español. Sin embargo, no se aplicó una limpieza demasiado agresiva, ya que cada fábula es relativamente corta y eliminar demasiadas palabras podía afectar la extracción de temas. Esta decisión permitió conservar términos importantes relacionados con personajes, acciones y objetos centrales de cada historia.

Posteriormente se aplicó LDA para obtener palabras clave de cada fábula. Debido a que cada documento representa una sola historia, se trabajó con un tópico por fábula. Las palabras clave obtenidas permitieron identificar elementos relevantes de cada relato, como personajes principales, conflictos y enseñanzas. También se observó que LDA es sensible a la calidad del texto de entrada, por lo que los errores de transcripción o limpieza pueden influir directamente en las palabras clave resultantes.

Finalmente, se utilizó un modelo LLM para transformar las palabras clave generadas por LDA en un resumen breve y tres posibles subtemas por fábula. Esta combinación resulta útil porque LDA identifica términos representativos, mientras que el LLM ayuda a convertir esos términos en información más comprensible e interpretable. Sin embargo, los resultados generados deben revisarse de forma crítica, ya que el LLM no reconstruye necesariamente la historia completa, sino que genera una síntesis basada en las palabras clave disponibles.

En conclusión, la actividad permitió integrar técnicas de audio-a-texto, limpieza de texto, modelado de tópicos y generación de lenguaje natural. El mayor aprendizaje fue observar que la calidad de las etapas iniciales, especialmente la transcripción y limpieza, afecta directamente los resultados posteriores de LDA y del LLM. Por ello, en problemas reales de NLP es fundamental revisar y depurar cuidadosamente los datos antes de generar análisis o conclusiones automáticas.


# **Fin de la actividad LDA y LMM: audio-a-texto**